# 03 - Grievance Status Analysis

Two parallel status views are used throughout:

* **`status_group`** -- four buckets used for all count / volume breakdowns:
  Discard, Disposed with Benefit, Disposed, Open.
* **`status_bucket`** -- four buckets used for disposal-time statistics:
  Discard, Disposed with Benefit, Disposed, In Follow Up with close

Breakdowns produced for both views:
yearly, district, office, department, category, subcategory.

Shared scaffolding lives in `grievance_common.py`.

## Setup

In [1]:
import numpy as np
import pandas as pd

from grievance_common import (
    load_complaints, add_time_fields, filter_window, compute_disposal_days,
    clean_series, status_bucket, status_group, OUTPUT_DIR, FOCUS_YEAR, VALID_YEARS,
)

In [2]:
import importlib
import grievance_common
importlib.reload(grievance_common)
from grievance_common import *

## Data Load and Preparation

`df_filtered` is the windowed complaints frame with the four-way
`status_group` (Discard / Disposed with Benefit / Disposed / Open) attached.
`df_res` is the same frame with `disposal_days` computed (negatives
dropped) and the four-way `status_bucket` (Discard / Disposed with Benefit /
Disposed / In Follow Up) attached -- this is the frame used for
disposal-time stats.

In [3]:
df = add_time_fields(load_complaints())

# Volume view: status_group (Discard / Disposed with Benefit / Disposed / Open)
df_filtered = filter_window(df)
df_filtered["status_group"] = status_group(
    df_filtered["status"], df_filtered["benefitted"])

# Resolution view: status_bucket (Discard / Disposed with Benefit / Disposed
# / In Follow Up With Close). disposal_days computed; negatives dropped.
df_res = filter_window(compute_disposal_days(df, drop_negative=True))
df_res["status_bucket"] = status_bucket(
    df_res["status"], df_res["benefitted"])

print(f"Volume frame   (df_filtered): {len(df_filtered):,} rows")
print(f"Resolution frm (df_res)     : {len(df_res):,} rows")
print("\nstatus_group value counts (df_filtered):")
print(df_filtered["status_group"].value_counts())
print("\nstatus_bucket value counts (df_res):")
print(df_res["status_bucket"].value_counts())

Volume frame   (df_filtered): 1,323,840 rows
Resolution frm (df_res)     : 1,323,827 rows

status_group value counts (df_filtered):
status_group
Disposed                 911613
Discard                  187891
Open                     132560
Disposed with Benefit     91776
Name: count, dtype: int64

status_bucket value counts (df_res):
status_bucket
Disposed                   911600
Discard                    187891
Disposed with Benefit       91776
In Follow Up With Close        81
Name: count, dtype: int64


## Helper Functions

Two helpers that produce a `level x status` pivot table -- one for counts
(any year, with within-level percentages) and one for disposal-time stats
(mean / median days). Reused across every breakdown below.

In [4]:
STATUS_GROUP_ORDER  = ["Discard", "Disposed with Benefit", "Disposed", "Open"]
STATUS_BUCKET_ORDER = ["Discard", "Disposed with Benefit", "Disposed",
                       "In Follow Up With Close"]
YEAR_ORDER = VALID_YEARS + ["Aggregate"]


def _add_aggregate(df, year_col="custom_year"):
    """Append a copy of `df` with year set to 'Aggregate'."""
    agg = df.copy()
    agg[year_col] = "Aggregate"
    return pd.concat([df, agg], ignore_index=True)


def status_count_pivot(data, level_col,
                       status_col="status_group",
                       status_order=STATUS_GROUP_ORDER,
                       ticket_col="ticket_no",
                       year_col="custom_year"):
    """level x (year x status -> count, percentage).

    When level_col IS the year column the yearly breakdown is returned
    directly (no nested year grouping). Otherwise columns are grouped by
    year first (including an 'Aggregate' group spanning all years) and
    then by status, so you see both the overall picture and each year
    side by side.

    Percentage = share of that status within (level, year).
    """
    d = data.copy()
    d[level_col] = clean_series(d[level_col])

    is_yearly = (level_col == year_col)

    if is_yearly:
        # Simple: aggregate only needs the one Aggregate row appended.
        d = _add_aggregate(d, year_col)
        long = (
            d.groupby([level_col, status_col])[ticket_col]
             .nunique().reset_index(name="count")
        )
        long["denom"] = (long.groupby(level_col)["count"].transform("sum"))
        long["percentage"] = (long["count"] / long["denom"] * 100).round(2)
        long = long.drop(columns="denom")

        pivot = (
            long.pivot(index=level_col, columns=status_col,
                       values=["count", "percentage"])
                .swaplevel(0, 1, axis=1).sort_index(axis=1).fillna(0)
        )
        ordered = [(s, m) for s in status_order
                   for m in ("count", "percentage")
                   if (s, m) in pivot.columns]
        return pivot.reindex(
            columns=pd.MultiIndex.from_tuples(ordered))

    # Non-yearly: group by (level, year, status) — with Aggregate appended.
    d = _add_aggregate(d, year_col)
    long = (
        d.groupby([level_col, year_col, status_col])[ticket_col]
         .nunique().reset_index(name="count")
    )
    long["denom"] = (long.groupby([level_col, year_col])["count"]
                         .transform("sum"))
    long["percentage"] = (long["count"] / long["denom"] * 100).round(2)
    long = long.drop(columns="denom")

    pivot = long.pivot_table(
        index=level_col,
        columns=[year_col, status_col],
        values=["count", "percentage"],
        aggfunc="sum"
    )
    # Flatten MultiIndex from (metric, year, status) -> (year, status, metric)
    pivot.columns = pd.MultiIndex.from_tuples(
        [(yr, st, m) for m, yr, st in pivot.columns]
    )
    ordered = [
        (yr, st, m)
        for yr in YEAR_ORDER
        for st in status_order
        for m in ("count", "percentage")
        if (yr, st, m) in pivot.columns
    ]
    return pivot.reindex(
        columns=pd.MultiIndex.from_tuples(ordered)).fillna(0)


def disposal_time_pivot(data, level_col,
                        status_col="status_bucket",
                        status_order=STATUS_BUCKET_ORDER,
                        ticket_col="ticket_no",
                        year_col="custom_year"):
    """level x (year x status -> count, %, mean days, median days).

    Same aggregate + year-wise layout as status_count_pivot.
    """
    d = data.copy()
    d[level_col] = clean_series(d[level_col])

    is_yearly = (level_col == year_col)

    if is_yearly:
        d = _add_aggregate(d, year_col)
        agg = (
            d.groupby([level_col, status_col])
             .agg(count=(ticket_col, "nunique"),
                  mean=("disposal_days", "mean"),
                  median=("disposal_days", "median"))
             .reset_index()
        )
        agg[["mean", "median"]] = agg[["mean", "median"]].round(0)
        agg["denom"] = agg.groupby(level_col)["count"].transform("sum")
        agg["percentage"] = (agg["count"] / agg["denom"] * 100).round(2)
        agg = agg.drop(columns="denom")

        pivot = (
            agg.pivot(index=level_col, columns=status_col,
                      values=["count", "percentage", "mean", "median"])
               .swaplevel(0, 1, axis=1).sort_index(axis=1).fillna(0)
        )
        metrics = ["count", "percentage", "mean", "median"]
        ordered = [(s, m) for s in status_order for m in metrics
                   if (s, m) in pivot.columns]
        return pivot.reindex(columns=pd.MultiIndex.from_tuples(ordered))

    d = _add_aggregate(d, year_col)
    agg = (
        d.groupby([level_col, year_col, status_col])
         .agg(count=(ticket_col, "nunique"),
              mean=("disposal_days", "mean"),
              median=("disposal_days", "median"))
         .reset_index()
    )
    agg[["mean", "median"]] = agg[["mean", "median"]].round(0)
    agg["denom"] = (agg.groupby([level_col, year_col])["count"]
                       .transform("sum"))
    agg["percentage"] = (agg["count"] / agg["denom"] * 100).round(2)
    agg = agg.drop(columns="denom")

    pivot = agg.pivot_table(
        index=level_col,
        columns=[year_col, status_col],
        values=["count", "percentage", "mean", "median"],
        aggfunc="sum"
    )
    pivot.columns = pd.MultiIndex.from_tuples(
        [(yr, st, m) for m, yr, st in pivot.columns]
    )
    metrics = ["count", "percentage", "mean", "median"]
    ordered = [
        (yr, st, m)
        for yr in YEAR_ORDER
        for st in status_order
        for m in metrics
        if (yr, st, m) in pivot.columns
    ]
    return pivot.reindex(
        columns=pd.MultiIndex.from_tuples(ordered)).fillna(0)

In [16]:
# =============================================================================
# Petitioner-level Helper Function
# =============================================================================
 
def _add_petitioner_id(data):
    """Add petitioner_id (name | district | block) column."""
    d = data.copy()
    for col in ["petitioner_name", "district", "block"]:
        d[col] = d[col].fillna("NaN").astype(str).str.strip().replace("", "NaN")
    d["petitioner_id"] = (d["petitioner_name"] + " | "
                          + d["district"] + " | " + d["block"])
    return d
 
 
def petitioner_status_count_pivot(data, level_col,
                                  status_col="status_group",
                                  status_order=STATUS_GROUP_ORDER,
                                  year_col="custom_year"):
    """Exactly like status_count_pivot but counts unique petitioners
    instead of unique tickets. Uses petitioner_id as the count column."""
    d = _add_petitioner_id(data)
    return status_count_pivot(d, level_col,
                              status_col=status_col,
                              status_order=status_order,
                              ticket_col="petitioner_id",
                              year_col=year_col)

def petitioner_disposal_time_pivot(data, level_col,
                                   status_col="status_bucket",
                                   status_order=STATUS_BUCKET_ORDER,
                                   year_col="custom_year"):
    """Exactly like disposal_time_pivot but counts unique petitioners
    instead of unique tickets. Uses petitioner_id as the count column."""
    d = _add_petitioner_id(data)
    return disposal_time_pivot(d, level_col,
                               status_col=status_col,
                               status_order=status_order,
                               ticket_col="petitioner_id",
                               year_col=year_col)

In [5]:
pd.set_option("display.max_columns", None)

## 1. Status Group Counts

Count and percentage for the four-way status group (Discard / Disposed with Benefit / Disposed / Open) at each breakdown level. Columns are grouped first by year (plus an **Aggregate** column spanning all years), then by status.

### Yearly

In [6]:
pivot_status_yearly = status_count_pivot(df_filtered, "custom_year")
pivot_status_yearly

Discard            Disposed with Benefit             Disposed  \
                count percentage                 count percentage     count   
custom_year                                                                   
2021-2022     13061.0      13.29               11613.0      11.82   71583.0   
2022-2023     28687.0      15.79               24948.0      13.73  123123.0   
2023-2024     39315.0      11.06               30033.0       8.45  267822.0   
2024-2025    106828.0      15.52               25182.0       3.66  449085.0   
Aggregate    187891.0      14.19               91776.0       6.93  911613.0   

                            Open             
            percentage     count percentage  
custom_year                                  
2021-2022        72.85    2001.0       2.04  
2022-2023        67.76    4944.0       2.72  
2023-2024        75.32   18409.0       5.18  
2024-2025        65.25  107206.0      15.58  
Aggregate        68.86  132560.0      10.01

### District

In [14]:
pivot_status_district = status_count_pivot(df_filtered, "district")
pivot_status_district

2021-2022                                                       \
                 Discard            Disposed with Benefit            Disposed   
                   count percentage                 count percentage    count   
district                                                                        
Angul                476      17.25                   245       8.88     2011   
Balangir             458      11.19                   958      23.40     2622   
Baleswar            1171      15.11                   348       4.49     6084   
Bargarh              537      15.67                   247       7.21     2602   
Bhadrak              649      13.13                   242       4.90     3830   
Boudh                191      24.55                   103      13.24      478   
Cuttack              689      16.06                   430      10.02     3078   
Deogarh               70      11.78                    38       6.40      485   
Dhenkanal            992      22.91                   140       3.23     3175   
Gajapati             137       9.62                    66       4.63     1206   
Ganjam               719      14.35                   508      10.14     3736   
Jagatsinghapur       323       9.88                   152       4.65     2765   
Jajpur              1126      18.64                   228       3.77     4032   
Jharsuguda           118      15.90                    87      11.73      509   
Kalahandi            917       9.75                   652       6.93     7818   
Kandhamal            208       8.51                   348      14.24     1859   
Kendrapara           432       8.91                   274       5.65     4099   
Kendujhar            198       9.51                   179       8.60     1642   
Khordha             1087      13.56                  2855      35.62     3939   
Koraput              212      15.45                   192      13.99      951   
Malkangiri            80      16.95                    90      19.07      293   
Mayurbhanj           236       8.39                  1223      43.49     1309   
NaN                  140       3.62                   603      15.57     3042   
Nabarangpur          121      11.77                   106      10.31      798   
Nayagarh             297      12.07                   198       8.05     1953   
Nuapada              116       9.29                    95       7.61     1025   
Puri                 452      14.15                   350      10.95     2331   
Rayagada             159      13.58                   148      12.64      842   
Sambalpur            274      17.42                   136       8.65     1136   
Subarnapur           182      14.44                   223      17.70      840   
Sundargarh           294      18.97                   149       9.61     1093   

                                           2022-2023             \
                           Open              Discard              
               percentage count percentage     count percentage   
district                                                          
Angul               72.89    27       0.98       651      17.84   
Balangir            64.04    56       1.37      1990      16.01   
Baleswar            78.51   146       1.88      3255      20.55   
Bargarh             75.95    40       1.17       736      16.28   
Bhadrak             77.48   222       4.49      1179      14.18   
Boudh               61.44     6       0.77       597      13.93   
Cuttack             71.73    94       2.19      1272      16.75   
Deogarh             81.65     1       0.17       115      12.18   
Dhenkanal           73.33    23       0.53      1960      33.48   
Gajapati            84.69    15       1.05       562      14.06   
Ganjam              74.57    47       0.94      2042      16.18   
Jagatsinghapur      84.61    28       0.86       555       6.96   
Jajpur              66.73   656      10.86      2347      23.74   
Jharsuguda          68.60    28       3.

### Office

In [15]:
pivot_status_office = status_count_pivot(df_filtered, "office")
pivot_status_office

2021-2022                                   \
                           Discard            Disposed with Benefit   
                             count percentage                 count   
office                                                                
Chief Secretary              702.0      25.79                1501.0   
Collector                   4442.0      13.53                5305.0   
DG & IG Police                83.0       4.07                 110.0   
Departments                 3654.0      18.53                2208.0   
Governor                     340.0      19.54                   3.0   
NaN                          193.0       2.01                1471.0   
Office of Chief Minister    3092.0      13.23                 780.0   
Superintendent of Police     555.0       8.94                 235.0   

                                                                           \
                                    Disposed              Open              
                         percentage    count percentage  count percentage   
office                                                                      
Chief Secretary               55.14    454.0      16.68   65.0       2.39   
Collector                     16.16  22910.0      69.78  176.0       0.54   
DG & IG Police                 5.40   1817.0      89.16   28.0       1.37   
Departments                   11.20  13532.0      68.61  329.0       1.67   
Governor                       0.17   1358.0      78.05   39.0       2.24   
NaN                           15.29   7729.0      80.32  230.0       2.39   
Office of Chief Minister       3.34  19222.0      82.26  274.0       1.17   
Superintendent of Police       3.78   4561.0      73.43  860.0      13.85   

                         2022-2023                                   \
                           Discard            Disposed with Benefit   
                             count percentage                 count   
office                                                                
Chief Secretary             1737.0      27.53                3814.0   
Collector                  10746.0      12.78               15733.0   
DG & IG Police               224.0       5.83                   0.0   
Departments                 6748.0      19.94                1767.0   
Governor                     580.0      28.40                   0.0   
NaN                          167.0       1.60                1380.0   
Office of Chief Minister    7836.0      24.56                1283.0   
Superintendent of Police     649.0       7.02                 971.0   

                                                                            \
                                    Disposed               Open              
                         percentage    count percentage   count percentage   
office                                                                       
Chief Secretary               60.45    515.0       8.16   243.0       3.85   
Collector                     18.71  56507.0      67.21  1085.0       1.29   
DG & IG Police                 0.00   3514.0      91.53   101.0       2.63   
Departments                    5.22  24299.0      71.79  1033.0       3.05   
Governor                       0.00   1390.0      68.07    72.0       3.53   
NaN                           13.22   8340.0      79.90   551.0       5.28   
Office of Chief Minister       4.02  22326.0      69.97   462.0       1.45   
Superintendent of Police      10.50   6232.0      67.38  1397.0      15.10   

                         2023-2024                                   \
                           Discard            Disposed with Benefit   
                             count percentage                 count   
office                                                                
Chief Secretary             3353.0      37.03                5003.0   
Collector                  13928.0       6.23               20316.0   
DG & IG Police               37

### Department

In [16]:
pivot_status_dept = status_count_pivot(df_filtered, "dept")
pivot_status_dept

2021-2022             \
                                                     Discard              
                                                       count percentage   
dept                                                                      
Agriculture & Farmers' Empowerment                      75.0       3.47   
Co-operation                                             7.0       2.24   
Commerce & Transport                                    12.0       3.08   
Electronics & Information Technology                     4.0       3.85   
Energy                                                  13.0       0.81   
Excise                                                   2.0       0.97   
Finance                                                 13.0       5.20   
Fisheries & Animal Resources Development                 1.0       0.42   
Food Supplies & Consumer Welfare                        43.0       1.61   
Forest, Environment and Climate Change Department        7.0       2.22   
General Administration & Public Grievance              176.0       7.90   
Handlooms ,Textiles & Handicrafts                        1.0       3.03   
Health & Family Welfare                                 81.0       1.84   
Higher Education                                        70.0      12.22   
Home department                                        179.0       1.81   
Housing & Urban Development                            131.0       1.93   
Industries                                               5.0       7.46   
Information & Public Relations                          10.0      15.62   
Labour & Employees' State Insurance                     12.0       2.61   
Law                                                      2.0       1.85   
Micro, Small & Medium Enterprise                        13.0      22.03   
Mission Shakti                                           0.0       0.00   
NaN                                                  11409.0      40.35   
Odia Language Literature & Culture                       2.0       1.45   
Panchayati Raj & Drinking Water                        289.0       1.36   
Parliamentary Affairs                                    0.0       0.00   
Planning & Convergence                                   1.0       2.27   
Public Enterprises                                       0.0       0.00   
Revenue & Disaster Management                          168.0       2.84   
Rural Development                                       56.0       8.63   
ST & SC Development, Minorities & Backward Clas...      54.0       8.71   
School & Mass Education                                 66.0       1.74   
Science & Technology                                     0.0       0.00   
Skill Development & Technical Education                 10.0       4.26   
Social Security & Empowerment of Persons with D...      73.0       3.51   
Sports & Youth Services                                  1.0       1.96   
Steel & Mines                                            6.0       8.33   
Tourism                                                  3.0       7.32   
Water Resources                                         15.0       1.92   
Women & Child Development                                7.0       0.94   
Works                                                   44.0       9.28   

                                                                          \
                                                   Disposed with Benefit   
                                                                   count   
dept                                                                       
Agriculture & Farmers' Empowerment                                 162.0   
Co-operation                                                        19.0   
Commerce & Transport                                                31.0   
Electronics & Information Technology                                 7.0   
Energy                                                             

### Category

In [17]:
pivot_status_category = status_count_pivot(df_filtered, "category")
pivot_status_category

2021-2022                                   \
                              Discard            Disposed with Benefit   
                                count percentage                 count   
category                                                                 
Accident                          1.0       7.69                   3.0   
Agriculture & Farming            25.0       2.43                  77.0   
BSKY                              0.0       0.00                   4.0   
CMRF                              3.0       0.58                 165.0   
COVID-19                          6.0       3.55                  31.0   
Culture                           0.0       0.00                   1.0   
Disaster Management               0.0       0.00                  24.0   
Education                        66.0       6.23                  92.0   
Energy                           26.0       1.36                 512.0   
Environment                       0.0       0.00                   7.0   
Excise                            1.0       0.52                   4.0   
Financial Assistance             17.0       2.29                  94.0   
General                         268.0       2.15                2723.0   
Health Care                      53.0       2.04                 221.0   
Housing                         161.0       1.19                 948.0   
ICDS                              2.0       0.91                   5.0   
Infrastructure                  117.0       4.55                 334.0   
Irrigation                        0.0       0.00                   0.0   
Land Matters                     68.0       2.55                 331.0   
Legal                             0.0       0.00                   1.0   
Miscellaneous                   157.0       4.04                 658.0   
NaN                           11621.0      37.51                1756.0   
Pension/Retirement Benefits      30.0       2.34                 695.0   
Police Case                      99.0       1.66                1090.0   
Public Utility                   24.0       2.29                  84.0   
Scheme &amp;amp; Benefits         0.0       0.00                   0.0   
School & College                  7.0       2.35                  14.0   
Service Matters                 118.0       3.32                 382.0   
Social Welfare                  148.0       1.89                 585.0   
Sports                            1.0       2.38                   1.0   
Tourism                           0.0       0.00                   0.0   
Traffic                           0.0       0.00                   4.0   
Transport                         6.0       2.52                  12.0   
Waste Management                  9.0       3.44                  94.0   
Water Supply                     26.0       0.95                 653.0   
Women Issues                      1.0       1.03                   8.0   

                                                                              \
                                       Disposed              Open              
                            percentage    count percentage  count percentage   
category                                                                       
Accident                         23.08      9.0      69.23    0.0       0.00   
Agriculture & Farming             7.50    912.0      88.80   13.0       1.27   
BSKY                              6.15     51.0      78.46   10.0      15.38   
CMRF                             31.73    349.0      67.12    3.0       0.58   
COVID-19                         18.34    127.0      75.15    5.0       2.96   
Culture                           7.69     12.0      92.31    0.0       0.00   
Disaster Management              27.91     62.0      72.09    0.0       0.00   
Education                         8.68    859.0      81.04   43.0       4.06   
Energy                           26.76   1345.0      70.31   30.0       1.57   
Environment                

### Subcategory

In [18]:
pivot_status_subcategory = status_count_pivot(df_filtered, "subcategory")
pivot_status_subcategory

2021-2022                                   \
                                Discard            Disposed with Benefit   
                                  count percentage                 count   
subcategory                                                                
AIDS                                0.0       0.00                   0.0   
Aadhar Card Issues                  0.0       0.00                   1.0   
Absence of officials                0.0       0.00                   0.0   
Absurd/Irrelevant                   0.0       0.00                   1.0   
Accidental Claims                   0.0       0.00                   0.0   
...                                 ...        ...                   ...   
Voltage fluctuations                0.0       0.00                   3.0   
Water Supply issue                 11.0       4.47                  74.0   
Water Tanks/ Reservoirs/Ponds       0.0       0.00                   0.0   
Web Portal Issues                   0.0       0.00                   3.0   
Workplace Harrassement              0.0       0.00                   0.0   

                                                                               \
                                         Disposed             Open              
                              percentage    count percentage count percentage   
subcategory                                                                     
AIDS                                0.00      0.0       0.00   0.0       0.00   
Aadhar Card Issues                  5.88     15.0      88.24   1.0       5.88   
Absence of officials                0.00      0.0       0.00   0.0       0.00   
Absurd/Irrelevant                  20.00      4.0      80.00   0.0       0.00   
Accidental Claims                   0.00      5.0     100.00   0.0       0.00   
...                                  ...      ...        ...   ...        ...   
Voltage fluctuations               15.79     16.0      84.21   0.0       0.00   
Water Supply issue                 30.08    154.0      62.60   7.0       2.85   
Water Tanks/ Reservoirs/Ponds       0.00      0.0       0.00   0.0       0.00   
Web Portal Issues                  20.00     12.0      80.00   0.0       0.00   
Workplace Harrassement              0.00      8.0     100.00   0.0       0.00   

                              2022-2023                                   \
                                Discard            Disposed with Benefit   
                                  count percentage                 count   
subcategory                                                                
AIDS                                0.0       0.00                   0.0   
Aadhar Card Issues                  0.0       0.00                   4.0   
Absence of officials                0.0       0.00                   0.0   
Absurd/Irrelevant                   0.0       0.00                   1.0   
Accidental Claims                   0.0       0.00                   5.0   
...                                 ...        ...                   ...   
Voltage fluctuations                0.0       0.00                   2.0   
Water Supply issue                 10.0       3.47                  79.0   
Water Tanks/ Reservoirs/Ponds       0.0       0.00                   0.0   
Web Portal Issues                   1.0       7.14                   1.0   
Workplace Harrassement              0.0       0.00                   0.0   

                                                                               \
                                         Disposed             Open              
                              percentage    count percentage count percentage   
subcategory                                                                     
AIDS                                0.00      1.0     100.00   0.0       0.00   
Aadhar Card Issues                 28.57      9.0      64.29   1.0       7.14   
Absence of officials                0.00      

### Export -- Status Group Tables

In [ ]:
output_file = f"{OUTPUT_DIR}/status_group_breakdowns.xlsx"

with pd.ExcelWriter(output_file) as writer:
    pivot_status_yearly.to_excel(writer, sheet_name="Yearly")
    pivot_status_district.to_excel(writer, sheet_name="District")
    pivot_status_office.to_excel(writer, sheet_name="Office")
    pivot_status_dept.to_excel(writer, sheet_name="Department")
    pivot_status_category.to_excel(writer, sheet_name="Category")
    pivot_status_subcategory.to_excel(writer, sheet_name="Subcategory")

print(f"Status-group breakdowns exported: {output_file}")

## 2. Disposal Time by Status Bucket

Count, percentage, mean and median disposal days (Discard / Disposed with Benefit / Disposed / In Follow Up) at each breakdown level, with the same Aggregate + year-wise layout.

### Yearly

In [5]:
pivot_res_yearly = disposal_time_pivot(df_res, "custom_year")
pivot_res_yearly

Discard                         Disposed with Benefit  \
                count percentage  mean median                 count   
custom_year                                                           
2021-2022     13061.0      13.56  67.0    3.0               11613.0   
2022-2023     28687.0      16.23  20.0    2.0               24948.0   
2023-2024     39315.0      11.66  21.0    3.0               30033.0   
2024-2025    106828.0      18.38  30.0    3.0               25182.0   
Aggregate    187891.0      15.77  29.0    3.0               91776.0   

                                      Disposed                           \
            percentage   mean median     count percentage   mean median   
custom_year                                                               
2021-2022        12.06  194.0  126.0   71583.0      74.34  219.0  163.0   
2022-2023        14.11   90.0   37.0  123120.0      69.65  107.0   48.0   
2023-2024         8.91   98.0   49.0  267814.0      79.43  109.0   64.0   
2024-2025         4.33   76.0   46.0  449083.0      77.28   63.0   41.0   
Aggregate         7.70  102.0   50.0  911600.0      76.52   95.0   51.0   

            In Follow Up With Close                           
                              count percentage   mean median  
custom_year                                                   
2021-2022                      38.0       0.04   71.0   35.0  
2022-2023                       6.0       0.00   21.0    7.0  
2023-2024                       2.0       0.00  439.0  439.0  
2024-2025                      35.0       0.01  144.0  181.0  
Aggregate                      81.0       0.01  108.0   57.0

### District

In [8]:
pivot_res_district = disposal_time_pivot(df_res, "district")
pivot_res_district

2021-2022                                                 \
                 Discard                          Disposed with Benefit   
                   count percentage   mean median                 count   
district                                                                  
Angul              476.0      17.42   66.0    3.0                 245.0   
Balangir           458.0      11.34   72.0    8.0                 958.0   
Baleswar          1171.0      15.40   43.0    4.0                 348.0   
Bargarh            537.0      15.84   57.0    4.0                 247.0   
Bhadrak            649.0      13.75  136.0    9.0                 242.0   
Boudh              191.0      24.74   42.0   10.0                 103.0   
Cuttack            689.0      16.42  117.0   22.0                 430.0   
Deogarh             70.0      11.80   72.0   12.0                  38.0   
Dhenkanal          992.0      23.03   16.0    1.0                 140.0   
Gajapati           137.0       9.72  108.0    6.0                  66.0   
Ganjam             719.0      14.49   77.0    3.0                 508.0   
Jagatsinghapur     323.0       9.97   71.0   16.0                 152.0   
Jajpur            1126.0      20.91   42.0    2.0                 228.0   
Jharsuguda         118.0      16.12   41.0    4.0                  87.0   
Kalahandi          917.0       9.77   41.0    2.0                 652.0   
Kandhamal          208.0       8.61   67.0   20.0                 348.0   
Kendrapara         432.0       8.99   63.0    4.0                 274.0   
Kendujhar          198.0       9.81   86.0    6.0                 179.0   
Khordha           1087.0      13.79   80.0    3.0                2855.0   
Koraput            212.0      15.63   76.0   14.0                 192.0   
Malkangiri          80.0      17.28   30.0    2.0                  90.0   
Mayurbhanj         236.0       8.53   81.0    5.0                1223.0   
NaN                140.0       3.70  264.0  177.0                 603.0   
Nabarangpur        121.0      11.80   45.0    5.0                 106.0   
Nayagarh           297.0      12.12   67.0    3.0                 198.0   
Nuapada            116.0       9.39   48.0   10.0                  95.0   
Puri               452.0      14.43   76.0    6.0                 350.0   
Rayagada           159.0      13.84   72.0    4.0                 148.0   
Sambalpur          274.0      17.71   69.0    4.0                 136.0   
Subarnapur         182.0      14.61   70.0    8.0                 223.0   
Sundargarh         294.0      19.13   61.0    5.0                 149.0   

                                                                           \
                                        Disposed                            
               percentage   mean median    count percentage   mean median   
district                                                                    
Angul                8.96  179.0  150.0   2011.0      73.58  204.0  169.0   
Balangir            23.72  238.0  218.0   2622.0      64.92  295.0  272.0   
Baleswar             4.58  289.0  210.0   6084.0      80.01  280.0  229.0   
Bargarh              7.28  139.0   84.0   2602.0      76.73  146.0   97.0   
Bhadrak              5.13  353.0  232.0   3830.0      81.13  233.0  198.0   
Boudh               13.34  150.0   96.0    478.0      61.92  162.0  105.0   
Cuttack             10.25  298.0  259.0   3078.0      73.34  305.0  263.0   
Deogarh              6.41  189.0  150.0    485.0      81.79  256.0  215.0   
Dhenkanal            3.25  182.0  115.0   3175.0      73.70  119.0   59.0   
Gajapati             4.68  205.0  140.0   1206.0      85.59  212.0  113.0   
Ganjam              10.24  213.0  144.0   3736.0      75.28  178.0  106.0   
Jagatsinghapur       4.69  236.0  220.0   2765.0      85.31  205.0  158.0   
Jajpur               4.23  270.0  219.0   4032.0      74.86  200.0  145.0   
Jharsuguda          11.89  173.0   73.0    509.0      69.54  144.0   54.0   
Kal

### Office

In [9]:
pivot_res_office = disposal_time_pivot(df_res, "office")
pivot_res_office

2021-2022                           \
                           Discard                            
                             count percentage   mean median   
office                                                        
Chief Secretary              702.0      26.42   24.0    1.0   
Collector                   4442.0      13.59   39.0    3.0   
DG & IG Police                83.0       4.13   95.0   73.0   
Departments                 3654.0      18.83  111.0   39.0   
Governor                     340.0      19.99   41.0    4.0   
NaN                          193.0       2.05  325.0  268.0   
Office of Chief Minister    3092.0      13.39   19.0    1.0   
Superintendent of Police     555.0      10.37  239.0  126.0   

                                                                         \
                         Disposed with Benefit                            
                                         count percentage   mean median   
office                                                                    
Chief Secretary                         1501.0      56.49  306.0  222.0   
Collector                               5305.0      16.23  152.0  104.0   
DG & IG Police                           110.0       5.47   84.0   56.0   
Departments                             2208.0      11.38  182.0  149.0   
Governor                                   3.0       0.18  127.0  107.0   
NaN                                     1471.0      15.66  137.0   87.0   
Office of Chief Minister                 780.0       3.38  396.0  392.0   
Superintendent of Police                 235.0       4.39  281.0   64.0   

                                                            \
                         Disposed                            
                            count percentage   mean median   
office                                                       
Chief Secretary             454.0      17.09  250.0  210.0   
Collector                 22910.0      70.10  162.0  107.0   
DG & IG Police             1817.0      90.35  156.0  106.0   
Departments               13532.0      69.74  249.0  194.0   
Governor                   1358.0      79.84  287.0  213.0   
NaN                        7729.0      82.28  187.0  107.0   
Office of Chief Minister  19222.0      83.23  288.0  237.0   
Superintendent of Police   4561.0      85.19  181.0  143.0   

                                                                           \
                         In Follow Up With Close                            
                                           count percentage   mean median   
office                                                                      
Chief Secretary                              0.0       0.00    0.0    0.0   
Collector                                   24.0       0.07   55.0   33.0   
DG & IG Police                               1.0       0.05  149.0  149.0   
Departments                                 10.0       0.05  113.0   64.0   
Governor                                     0.0       0.00    0.0    0.0   
NaN                                          0.0       0.00    0.0    0.0   
Office of Chief Minister                     0.0       0.00    0.0    0.0   
Superintendent of Police                     3.0       0.06   27.0   14.0   

                         2022-2023                           \
                           Discard                            
                             count percentage   mean median   
office                                                        
Chief Secretary             1737.0      28.64    6.0    1.0   
Collector                  10746.0      12.95   11.0    1.0   
DG & IG Police               224.0       5.99   10.0    2.0   
Departments                 6748.0      20.56   38.0    3.0   
Governor                     580.0      29.44   18.0    7.0   
NaN                          167.0       1.69  282.0  168.0   
Office of Chief Minister    7836.0      24.92    3.0    1.0   
Supe

### Department

In [10]:
pivot_res_dept = disposal_time_pivot(df_res, "dept")
pivot_res_dept

2021-2022             \
                                                     Discard              
                                                       count percentage   
dept                                                                      
Agriculture & Farmers' Empowerment                      75.0       3.49   
Co-operation                                             7.0       2.28   
Commerce & Transport                                    12.0       3.08   
Electronics & Information Technology                     4.0       3.92   
Energy                                                  13.0       0.82   
Excise                                                   2.0       0.98   
Finance                                                 13.0       5.24   
Fisheries & Animal Resources Development                 1.0       0.42   
Food Supplies & Consumer Welfare                        43.0       1.61   
Forest, Environment and Climate Change Department        7.0       2.24   
General Administration & Public Grievance              176.0       8.02   
Handlooms ,Textiles & Handicrafts                        1.0       3.03   
Health & Family Welfare                                 81.0       1.93   
Higher Education                                        70.0      12.48   
Home department                                        179.0       2.00   
Housing & Urban Development                            131.0       1.95   
Industries                                               5.0       7.58   
Information & Public Relations                          10.0      15.62   
Labour & Employees' State Insurance                     12.0       2.63   
Law                                                      2.0       1.90   
Micro, Small & Medium Enterprise                        13.0      22.41   
Mission Shakti                                           0.0       0.00   
NaN                                                  11409.0      40.62   
Odia Language Literature & Culture                       2.0       1.46   
Panchayati Raj & Drinking Water                        289.0       1.37   
Parliamentary Affairs                                    0.0       0.00   
Planning & Convergence                                   1.0       2.27   
Public Enterprises                                       0.0       0.00   
Revenue & Disaster Management                          168.0       2.85   
Rural Development                                       56.0       8.79   
ST & SC Development, Minorities & Backward Clas...      54.0       8.87   
School & Mass Education                                 66.0       1.84   
Science & Technology                                     0.0       0.00   
Skill Development & Technical Education                 10.0       4.31   
Social Security & Empowerment of Persons with D...      73.0       3.53   
Sports & Youth Services                                  1.0       2.00   
Steel & Mines                                            6.0       8.45   
Tourism                                                  3.0       7.50   
Water Resources                                         15.0       2.01   
Women & Child Development                                7.0       0.97   
Works                                                   44.0       9.40   

                                                                  \
                                                                   
                                                     mean median   
dept                                                               
Agriculture & Farmers' Empowerment                   85.0   25.0   
Co-operation                                        251.0  243.0   
Commerce & Transport                                158.0  151.0   
Electronics & Information Technology                222.0  152.0   
Energy                                              188.0   69.0   
Excise                                               70.0   70.

### Category

In [11]:
pivot_res_category = disposal_time_pivot(df_res, "category")
pivot_res_category

2021-2022                           \
                              Discard                            
                                count percentage   mean median   
category                                                         
Accident                          1.0       7.69  243.0  243.0   
Agriculture & Farming            25.0       2.47  199.0   62.0   
BSKY                              0.0       0.00    0.0    0.0   
CMRF                              3.0       0.58  411.0  460.0   
COVID-19                          6.0       3.66  235.0  147.0   
Culture                           0.0       0.00    0.0    0.0   
Disaster Management               0.0       0.00    0.0    0.0   
Education                        66.0       6.48  530.0  546.0   
Energy                           26.0       1.38  200.0  166.0   
Environment                       0.0       0.00    0.0    0.0   
Excise                            1.0       0.52  121.0  121.0   
Financial Assistance             17.0       2.31  214.0  187.0   
General                         268.0       2.30  222.0  166.0   
Health Care                      53.0       2.11  375.0  348.0   
Housing                         161.0       1.19  125.0   80.0   
ICDS                              2.0       0.92   50.0   50.0   
Infrastructure                  117.0       4.68  200.0  105.0   
Irrigation                        0.0       0.00    0.0    0.0   
Land Matters                     68.0       2.57  153.0  132.0   
Legal                             0.0       0.00    0.0    0.0   
Miscellaneous                   157.0       4.13  180.0  120.0   
NaN                           11621.0      37.81   49.0    2.0   
Pension/Retirement Benefits      30.0       2.40  243.0  125.0   
Police Case                      99.0       1.73  130.0  101.0   
Public Utility                   24.0       2.33  195.0  141.0   
Scheme &amp;amp; Benefits         0.0       0.00    0.0    0.0   
School & College                  7.0       2.55  559.0  781.0   
Service Matters                 118.0       3.44  241.0  153.0   
Social Welfare                  148.0       1.90  181.0   98.0   
Sports                            1.0       2.44  243.0  243.0   
Tourism                           0.0       0.00    0.0    0.0   
Traffic                           0.0       0.00    0.0    0.0   
Transport                         6.0       2.52  201.0  194.0   
Waste Management                  9.0       3.45  241.0  271.0   
Water Supply                     26.0       0.96  254.0  216.0   
Women Issues                      1.0       1.05  201.0  201.0   

                                                                            \
                            Disposed with Benefit                            
                                            count percentage   mean median   
category                                                                     
Accident                                      3.0      23.08  148.0  174.0   
Agriculture & Farming                        77.0       7.59  232.0  193.0   
BSKY                                          4.0       7.27  166.0   42.0   
CMRF                                        165.0      31.91  294.0  286.0   
COVID-19                                     31.0      18.90  227.0   99.0   
Culture                                       1.0       7.69   58.0   58.0   
Disaster Management                          24.0      27.91  106.0  100.0   
Education                                    92.0       9.03  294.0  110.0   
Energy                                      512.0      27.18  131.0   80.0   
Environment                                   7.0       4.49  201.0   46.0   
Excise                                        4.0       2.09  151.0  179.0   
Financial Assistance                         94.0      12.77  276.0  264.0   
General                                    2723.0      23.40  152.0   79.0   
Health Care                                 221.0       8.80 

### Subcategory

In [12]:
pivot_res_subcategory = disposal_time_pivot(df_res, "subcategory")
pivot_res_subcategory

2021-2022                           \
                                Discard                            
                                  count percentage   mean median   
subcategory                                                        
AIDS                                0.0        0.0    0.0    0.0   
Aadhar Card Issues                  0.0        0.0    0.0    0.0   
Absence of officials                0.0        0.0    0.0    0.0   
Absurd/Irrelevant                   0.0        0.0    0.0    0.0   
Accidental Claims                   0.0        0.0    0.0    0.0   
...                                 ...        ...    ...    ...   
Voltage fluctuations                0.0        0.0    0.0    0.0   
Water Supply issue                 11.0        4.6  282.0  215.0   
Water Tanks/ Reservoirs/Ponds       0.0        0.0    0.0    0.0   
Web Portal Issues                   0.0        0.0    0.0    0.0   
Workplace Harrassement              0.0        0.0    0.0    0.0   

                                                                              \
                              Disposed with Benefit                            
                                              count percentage   mean median   
subcategory                                                                    
AIDS                                            0.0       0.00    0.0    0.0   
Aadhar Card Issues                              1.0       6.25  131.0  131.0   
Absence of officials                            0.0       0.00    0.0    0.0   
Absurd/Irrelevant                               1.0      20.00  152.0  152.0   
Accidental Claims                               0.0       0.00    0.0    0.0   
...                                             ...        ...    ...    ...   
Voltage fluctuations                            3.0      15.79  260.0   46.0   
Water Supply issue                             74.0      30.96  153.0  106.0   
Water Tanks/ Reservoirs/Ponds                   0.0       0.00    0.0    0.0   
Web Portal Issues                               3.0      20.00  123.0  100.0   
Workplace Harrassement                          0.0       0.00    0.0    0.0   

                                                                 \
                              Disposed                            
                                 count percentage   mean median   
subcategory                                                       
AIDS                               0.0       0.00    0.0    0.0   
Aadhar Card Issues                15.0      93.75  198.0  161.0   
Absence of officials               0.0       0.00    0.0    0.0   
Absurd/Irrelevant                  4.0      80.00  409.0  230.0   
Accidental Claims                  5.0     100.00  164.0  153.0   
...                                ...        ...    ...    ...   
Voltage fluctuations              16.0      84.21  272.0  154.0   
Water Supply issue               154.0      64.44  253.0  166.0   
Water Tanks/ Reservoirs/Ponds      0.0       0.00    0.0    0.0   
Web Portal Issues                 12.0      80.00  297.0  151.0   
Workplace Harrassement             8.0     100.00  212.0  135.0   

                                                                              \
                              In Follow Up With Close                          
                                                count percentage mean median   
subcategory                                                                    
AIDS                                              0.0        0.0  0.0    0.0   
Aadhar Card Issues                                0.0        0.0  0.0    0.0   
Absence of officials                              0.0        0.0  0.0    0.0   
Absurd/Irrelevant                                 0.0        0.0  0.0    0.0   
Accidental Claims                                 0.0        0.0  0.0    0.0   
...                                               ...        ...  ...    ...   
Vo

## 3. Status Group at Petitioner Level

In [ ]:
pivot_pet_status_yearly = petitioner_status_count_pivot(
    df_filtered, "custom_year")
print("\nPetitioner Status Group — Yearly:")
print(pivot_pet_status_yearly)


Petitioner Status Group — Yearly:
              Discard            Disposed with Benefit             Disposed  \
                count percentage                 count percentage     count   
custom_year                                                                   
2021-2022      8685.0      12.20                9298.0      13.06   51517.0   
2022-2023     17848.0      13.74               19599.0      15.09   88420.0   
2023-2024     25508.0       9.20               24823.0       8.96  212794.0   
2024-2025     58784.0      11.68               17562.0       3.49  350752.0   
Aggregate    105509.0      11.29               68113.0       7.29  667418.0   

                           Open             
            percentage    count percentage  
custom_year                                 
2021-2022        72.34   1717.0       2.41  
2022-2023        68.09   3989.0       3.07  
2023-2024        76.78  14022.0       5.06  
2024-2025        69.68  76287.0      15.15  
Aggregate        

In [9]:
pivot_pet_status_district = petitioner_status_count_pivot(
    df_filtered, "district")
print("\nPetitioner Status Group — District:")
pivot_pet_status_district


Petitioner Status Group — District:


2021-2022                                                       \
                 Discard            Disposed with Benefit            Disposed   
                   count percentage                 count percentage    count   
district                                                                        
Angul                293      14.45                   198       9.76     1511   
Balangir             357      10.48                   807      23.70     2194   
Baleswar             886      14.47                   260       4.25     4846   
Bargarh              355      15.56                   181       7.93     1714   
Bhadrak              401      11.03                   184       5.06     2866   
Boudh                139      26.38                    79      14.99      304   
Cuttack              469      15.93                   351      11.92     2039   
Deogarh               50      10.64                    30       6.38      389   
Dhenkanal            794      23.41                   113       3.33     2468   
Gajapati              96       7.89                    53       4.35     1054   
Ganjam               470      12.60                   411      11.02     2807   
Jagatsinghapur       248       9.65                   127       4.94     2176   
Jajpur               391       9.97                   157       4.00     2773   
Jharsuguda            65      15.15                    55      12.82      288   
Kalahandi            671       8.44                   565       7.11     6698   
Kandhamal             50      10.73                    80      17.17      325   
Kendrapara           268       7.10                   230       6.09     3239   
Kendujhar            146       9.57                   124       8.13     1207   
Khordha              783      12.15                  2594      40.24     2946   
Koraput              136      16.00                   155      18.24      545   
Malkangiri            41      15.47                    75      28.30      141   
Mayurbhanj           201       8.45                  1118      47.01     1017   
NaN                   93       7.33                   278      21.91      845   
Nabarangpur           78      10.96                    80      11.24      551   
Nayagarh             241      12.27                   139       7.08     1572   
Nuapada               82       8.22                    78       7.82      827   
Puri                 266      12.84                   256      12.36     1505   
Rayagada             107      12.80                   127      15.19      583   
Sambalpur            179      17.13                   105      10.05      741   
Subarnapur           128      14.00                   172      18.82      602   
Sundargarh           201      18.72                   116      10.80      744   

                                           2022-2023             \
                           Open              Discard              
               percentage count percentage     count percentage   
district                                                          
Angul               74.51    26       1.28       442      18.31   
Balangir            64.43    47       1.38      1444      14.97   
Baleswar            79.13   132       2.16      2253      18.53   
Bargarh             75.11    32       1.40       456      15.76   
Bhadrak             78.82   185       5.09       538      10.58   
Boudh               57.69     5       0.95       267       8.45   
Cuttack             69.26    85       2.89       761      16.05   
Deogarh             82.77     1       0.21        71       9.77   
Dhenkanal           72.76    17       0.50      1269      34.39   
Gajapati            86.61    14       1.15       478      13.61   
Ganjam              75.23    43       1.15      1098      12.20   
Jagatsinghapur      84.67    19       0.74       328       5.39   
Jajpur              70.70   601      15.32      1224      20.51   
Jharsuguda          67.13    21       4.

In [10]:
pivot_pet_status_office = petitioner_status_count_pivot(
    df_filtered, "office")
print("\nPetitioner Status Group — Office:")
pivot_pet_status_office


Petitioner Status Group — Office:


2021-2022                                   \
                           Discard            Disposed with Benefit   
                             count percentage                 count   
office                                                                
Chief Secretary              526.0      24.96                1127.0   
Collector                   3716.0      12.84                4913.0   
DG & IG Police                75.0       4.17                 100.0   
Departments                 2673.0      18.56                1742.0   
Governor                     242.0      17.32                   3.0   
NaN                          148.0       4.62                 739.0   
Office of Chief Minister    1959.0       9.80                 758.0   
Superintendent of Police     462.0       8.58                 214.0   

                                                                           \
                                    Disposed              Open              
                         percentage    count percentage  count percentage   
office                                                                      
Chief Secretary               53.49    396.0      18.79   58.0       2.75   
Collector                     16.98  20143.0      69.61  167.0       0.58   
DG & IG Police                 5.56   1600.0      88.89   25.0       1.39   
Departments                   12.10   9702.0      67.38  282.0       1.96   
Governor                       0.21   1115.0      79.81   37.0       2.65   
NaN                           23.08   2164.0      67.58  151.0       4.72   
Office of Chief Minister       3.79  17007.0      85.09  262.0       1.31   
Superintendent of Police       3.98   3920.0      72.84  786.0      14.60   

                         2022-2023                                   \
                           Discard            Disposed with Benefit   
                             count percentage                 count   
office                                                                
Chief Secretary             1118.0      28.51                2178.0   
Collector                   8194.0      11.31               13866.0   
DG & IG Police               173.0       5.85                   0.0   
Departments                 4725.0      19.82                1392.0   
Governor                     401.0      27.43                   0.0   
NaN                          142.0       4.15                 725.0   
Office of Chief Minister    5469.0      21.43                1183.0   
Superintendent of Police     536.0       7.05                 879.0   

                                                                            \
                                    Disposed               Open              
                         percentage    count percentage   count percentage   
office                                                                       
Chief Secretary               55.53    421.0      10.73   205.0       5.23   
Collector                     19.14  49465.0      68.30   902.0       1.25   
DG & IG Police                 0.00   2697.0      91.27    85.0       2.88   
Departments                    5.84  16820.0      70.56   901.0       3.78   
Governor                       0.00    997.0      68.19    64.0       4.38   
NaN                           21.20   2237.0      65.41   316.0       9.24   
Office of Chief Minister       4.63  18437.0      72.23   436.0       1.71   
Superintendent of Police      11.56   4919.0      64.67  1272.0      16.72   

                         2023-2024                                   \
                           Discard            Disposed with Benefit   
                             count percentage                 count   
office                                                                
Chief Secretary             2082.0      38.87                2758.0   
Collector                  11432.0       5.77               19031.0   
DG & IG Police               24

In [12]:
pivot_pet_status_dept = petitioner_status_count_pivot(
    df_filtered, "dept")
print("\nPetitioner Status Group — Department:")
pivot_pet_status_dept


Petitioner Status Group — Department:


2021-2022             \
                                                     Discard              
                                                       count percentage   
dept                                                                      
Agriculture & Farmers' Empowerment                      70.0       4.11   
Co-operation                                             7.0       2.76   
Commerce & Transport                                    12.0       3.57   
Electronics & Information Technology                     4.0       5.33   
Energy                                                  13.0       1.22   
Excise                                                   2.0       1.40   
Finance                                                 13.0       5.60   
Fisheries & Animal Resources Development                 1.0       0.52   
Food Supplies & Consumer Welfare                        42.0       1.83   
Forest, Environment and Climate Change Department        5.0       1.98   
General Administration & Public Grievance              156.0       8.49   
Handlooms ,Textiles & Handicrafts                        1.0       3.33   
Health & Family Welfare                                 78.0       2.43   
Higher Education                                        58.0      12.31   
Home department                                        172.0       2.14   
Housing & Urban Development                            123.0       2.18   
Industries                                               4.0       6.90   
Information & Public Relations                          10.0      17.24   
Labour & Employees' State Insurance                     12.0       3.02   
Law                                                      2.0       2.11   
Micro, Small & Medium Enterprise                         4.0      12.12   
Mission Shakti                                           0.0       0.00   
NaN                                                   7525.0      33.21   
Odia Language Literature & Culture                       2.0       1.53   
Panchayati Raj & Drinking Water                        250.0       1.57   
Parliamentary Affairs                                    0.0       0.00   
Planning & Convergence                                   1.0       2.50   
Public Enterprises                                       0.0       0.00   
Revenue & Disaster Management                          158.0       3.21   
Rural Development                                       42.0       8.64   
ST & SC Development, Minorities & Backward Clas...      49.0       9.42   
School & Mass Education                                 58.0       2.04   
Science & Technology                                     0.0       0.00   
Skill Development & Technical Education                  8.0       4.82   
Social Security & Empowerment of Persons with D...      69.0       4.57   
Sports & Youth Services                                  1.0       2.22   
Steel & Mines                                            6.0      10.34   
Tourism                                                  3.0       8.11   
Water Resources                                         14.0       2.39   
Women & Child Development                                6.0       1.13   
Works                                                   40.0      10.26   

                                                                          \
                                                   Disposed with Benefit   
                                                                   count   
dept                                                                       
Agriculture & Farmers' Empowerment                                 142.0   
Co-operation                                                        16.0   
Commerce & Transport                                                29.0   
Electronics & Information Technology                                 7.0   
Energy                                                             

In [13]:
pivot_pet_status_category = petitioner_status_count_pivot(
    df_filtered, "category")
print("\nPetitioner Status Group — Category:")
pivot_pet_status_category


Petitioner Status Group — Category:


2021-2022                                   \
                              Discard            Disposed with Benefit   
                                count percentage                 count   
category                                                                 
Accident                          1.0       7.69                   3.0   
Agriculture & Farming            24.0       2.82                  73.0   
BSKY                              0.0       0.00                   4.0   
CMRF                              3.0       0.58                 165.0   
COVID-19                          6.0       4.00                  28.0   
Culture                           0.0       0.00                   1.0   
Disaster Management               0.0       0.00                  24.0   
Education                        55.0       6.38                  83.0   
Energy                           26.0       2.17                 366.0   
Environment                       0.0       0.00                   6.0   
Excise                            1.0       0.75                   4.0   
Financial Assistance             17.0       2.44                  90.0   
General                         247.0       2.33                2412.0   
Health Care                      49.0       2.98                 208.0   
Housing                         145.0       1.22                 887.0   
ICDS                              2.0       1.12                   5.0   
Infrastructure                   92.0       5.71                 264.0   
Irrigation                        0.0       0.00                   0.0   
Land Matters                     66.0       2.94                 303.0   
Legal                             0.0       0.00                   1.0   
Miscellaneous                   144.0       4.49                 585.0   
NaN                            7699.0      31.48                1546.0   
Pension/Retirement Benefits      30.0       2.54                 667.0   
Police Case                      95.0       1.99                 865.0   
Public Utility                   24.0       3.35                  74.0   
Scheme &amp;amp; Benefits         0.0       0.00                   0.0   
School & College                  7.0       2.57                  13.0   
Service Matters                 111.0       3.83                 345.0   
Social Welfare                  128.0       2.37                 503.0   
Sports                            1.0       2.78                   1.0   
Tourism                           0.0       0.00                   0.0   
Traffic                           0.0       0.00                   4.0   
Transport                         6.0       2.75                  11.0   
Waste Management                  9.0       4.39                  73.0   
Water Supply                     23.0       1.64                 365.0   
Women Issues                      1.0       1.06                   7.0   

                                                                              \
                                       Disposed              Open              
                            percentage    count percentage  count percentage   
category                                                                       
Accident                         23.08      9.0      69.23    0.0       0.00   
Agriculture & Farming             8.57    743.0      87.21   12.0       1.41   
BSKY                              6.25     50.0      78.12   10.0      15.62   
CMRF                             31.98    345.0      66.86    3.0       0.58   
COVID-19                         18.67    111.0      74.00    5.0       3.33   
Culture                           8.33     11.0      91.67    0.0       0.00   
Disaster Management              27.91     62.0      72.09    0.0       0.00   
Education                         9.63    681.0      79.00   43.0       4.99   
Energy                           30.55    783.0      65.36   23.0       1.92   
Environment                

In [14]:
 
pivot_pet_status_subcategory = petitioner_status_count_pivot(
    df_filtered, "subcategory")
print("\nPetitioner Status Group — Subcategory:")
pivot_pet_status_subcategory.head(10)


Petitioner Status Group — Subcategory:


2021-2022                                              \
                        Discard            Disposed with Benefit              
                          count percentage                 count percentage   
subcategory                                                                   
AIDS                        0.0       0.00                   0.0       0.00   
Aadhar Card Issues          0.0       0.00                   1.0       6.25   
Absence of officials        0.0       0.00                   0.0       0.00   
Absurd/Irrelevant           0.0       0.00                   1.0      20.00   
Accidental Claims           0.0       0.00                   0.0       0.00   
Admissions                  1.0       3.33                   5.0      16.67   
Agriculture/Farming         3.0       2.01                  13.0       8.72   
Ahar Yojana                 0.0       0.00                   1.0      20.00   
Allegation/Corruption      19.0       3.60                  32.0       6.06   
Anganwadi Centres           0.0       0.00                   0.0       0.00   

                                                           2022-2023  \
                      Disposed             Open              Discard   
                         count percentage count percentage     count   
subcategory                                                            
AIDS                       0.0       0.00   0.0       0.00       0.0   
Aadhar Card Issues        14.0      87.50   1.0       6.25       0.0   
Absence of officials       0.0       0.00   0.0       0.00       0.0   
Absurd/Irrelevant          4.0      80.00   0.0       0.00       0.0   
Accidental Claims          5.0     100.00   0.0       0.00       0.0   
Admissions                24.0      80.00   0.0       0.00       0.0   
Agriculture/Farming      132.0      88.59   1.0       0.67       4.0   
Ahar Yojana                4.0      80.00   0.0       0.00       0.0   
Allegation/Corruption    442.0      83.71  35.0       6.63      28.0   
Anganwadi Centres          0.0       0.00   0.0       0.00       0.0   

                                                                            \
                                 Disposed with Benefit            Disposed   
                      percentage                 count percentage    count   
subcategory                                                                  
AIDS                        0.00                   0.0       0.00      1.0   
Aadhar Card Issues          0.00                   4.0      28.57      9.0   
Absence of officials        0.00                   0.0       0.00      0.0   
Absurd/Irrelevant           0.00                   1.0       7.14     11.0   
Accidental Claims           0.00                   5.0      22.73     17.0   
Admissions                  0.00                  49.0      40.83     66.0   
Agriculture/Farming         0.94                  40.0       9.43    370.0   
Ahar Yojana                 0.00                   2.0      33.33      4.0   
Allegation/Corruption       1.81                 167.0      10.80   1260.0   
Anganwadi Centres           0.00                   0.0       0.00      2.0   

                                                  2023-2024             \
                                  Open              Discard              
                      percentage count percentage     count percentage   
subcategory                                                              
AIDS                      100.00   0.0       0.00       1.0      25.00   
Aadhar Card Issues         64.29   1.0       7.14       1.0       3.70   
Absence of officials        0.00   0.0       0.00       0.0       0.00   
Absurd/Irrelevant          78.57   2.0      14.29       0.0       0.00   
Accidental Claims          77.27   0.0       0.00       0.0       0.00   
Admissions                 55.00   5.0       4.17       1.0       0.59   
Agriculture/Farming        87.26  10.0       2.36       8.0       1.42   

## 4. Disposal Time by Status Bucket at Petitioner Level

In [17]:
pivot_pet_bucket_yearly = petitioner_disposal_time_pivot(
    df_res, "custom_year")
pivot_pet_bucket_yearly

Discard                         Disposed with Benefit  \
                count percentage  mean median                 count   
custom_year                                                           
2021-2022      8685.0      12.49  67.0    3.0                9298.0   
2022-2023     17848.0      14.18  20.0    2.0               19599.0   
2023-2024     25508.0       9.69  21.0    3.0               24823.0   
2024-2025     58784.0      13.76  30.0    3.0               17562.0   
Aggregate    105509.0      12.54  29.0    3.0               68113.0   

                                      Disposed                           \
            percentage   mean median     count percentage   mean median   
custom_year                                                               
2021-2022        13.37  194.0  126.0   51517.0      74.09  219.0  163.0   
2022-2023        15.57   90.0   37.0   88418.0      70.25  107.0   48.0   
2023-2024         9.43   98.0   49.0  212786.0      80.87  109.0   64.0   
2024-2025         4.11   76.0   46.0  350750.0      82.12   63.0   41.0   
Aggregate         8.10  102.0   50.0  667408.0      79.35   95.0   51.0   

            In Follow Up With Close                           
                              count percentage   mean median  
custom_year                                                   
2021-2022                      31.0       0.04   71.0   35.0  
2022-2023                       5.0       0.00   21.0    7.0  
2023-2024                       2.0       0.00  439.0  439.0  
2024-2025                      35.0       0.01  144.0  181.0  
Aggregate                      73.0       0.01  108.0   57.0

In [19]:
pivot_pet_bucket_district = petitioner_disposal_time_pivot(
    df_res, "district")

pivot_pet_bucket_district

2021-2022                                                 \
                 Discard                          Disposed with Benefit   
                   count percentage   mean median                 count   
district                                                                  
Angul              293.0      14.63   66.0    3.0                 198.0   
Balangir           357.0      10.63   72.0    8.0                 807.0   
Baleswar           886.0      14.78   43.0    4.0                 260.0   
Bargarh            355.0      15.76   57.0    4.0                 181.0   
Bhadrak            401.0      11.62  136.0    9.0                 184.0   
Boudh              139.0      26.63   42.0   10.0                  79.0   
Cuttack            469.0      16.40  117.0   22.0                 351.0   
Deogarh             50.0      10.66   72.0   12.0                  30.0   
Dhenkanal          794.0      23.52   16.0    1.0                 113.0   
Gajapati            96.0       7.98  108.0    6.0                  53.0   
Ganjam             470.0      12.74   77.0    3.0                 411.0   
Jagatsinghapur     248.0       9.72   71.0   16.0                 127.0   
Jajpur             391.0      11.77   42.0    2.0                 157.0   
Jharsuguda          65.0      15.44   41.0    4.0                  55.0   
Kalahandi          671.0       8.46   41.0    2.0                 565.0   
Kandhamal           50.0      10.96   67.0   20.0                  80.0   
Kendrapara         268.0       7.17   63.0    4.0                 230.0   
Kendujhar          146.0       9.88   86.0    6.0                 124.0   
Khordha            783.0      12.38   80.0    3.0                2594.0   
Koraput            136.0      16.25   76.0   14.0                 155.0   
Malkangiri          41.0      15.95   30.0    2.0                  75.0   
Mayurbhanj         201.0       8.60   81.0    5.0                1118.0   
NaN                 93.0       7.65  264.0  177.0                 278.0   
Nabarangpur         78.0      11.00   45.0    5.0                  80.0   
Nayagarh           241.0      12.33   67.0    3.0                 139.0   
Nuapada             82.0       8.31   48.0   10.0                  78.0   
Puri               266.0      13.12   76.0    6.0                 256.0   
Rayagada           107.0      13.10   72.0    4.0                 127.0   
Sambalpur          179.0      17.45   69.0    4.0                 105.0   
Subarnapur         128.0      14.17   70.0    8.0                 172.0   
Sundargarh         201.0      18.93   61.0    5.0                 116.0   

                                                                           \
                                        Disposed                            
               percentage   mean median    count percentage   mean median   
district                                                                    
Angul                9.89  179.0  150.0   1511.0      75.44  204.0  169.0   
Balangir            24.03  238.0  218.0   2194.0      65.32  295.0  272.0   
Baleswar             4.34  289.0  210.0   4846.0      80.86  280.0  229.0   
Bargarh              8.03  139.0   84.0   1714.0      76.08  146.0   97.0   
Bhadrak              5.33  353.0  232.0   2866.0      83.05  233.0  198.0   
Boudh               15.13  150.0   96.0    304.0      58.24  162.0  105.0   
Cuttack             12.28  298.0  259.0   2039.0      71.32  305.0  263.0   
Deogarh              6.40  189.0  150.0    389.0      82.94  256.0  215.0   
Dhenkanal            3.35  182.0  115.0   2468.0      73.10  119.0   59.0   
Gajapati             4.41  205.0  140.0   1054.0      87.61  212.0  113.0   
Ganjam              11.14  213.0  144.0   2807.0      76.11  178.0  106.0   
Jagatsinghapur       4.98  236.0  220.0   2176.0      85.27  205.0  158.0   
Jajpur               4.73  270.0  219.0   2773.0      83.50  200.0  145.0   
Jharsuguda          13.06  173.0   73.0    288.0      68.41  144.0   54.0   
Kal

In [20]:
pivot_pet_bucket_office = petitioner_disposal_time_pivot(
    df_res, "office")
pivot_pet_bucket_office

2021-2022                           \
                           Discard                            
                             count percentage   mean median   
office                                                        
Chief Secretary              526.0      25.67   24.0    1.0   
Collector                   3716.0      12.91   39.0    3.0   
DG & IG Police                75.0       4.22   95.0   73.0   
Departments                 2673.0      18.92  111.0   39.0   
Governor                     242.0      17.79   41.0    4.0   
NaN                          148.0       4.85  325.0  268.0   
Office of Chief Minister    1959.0       9.93   19.0    1.0   
Superintendent of Police     462.0      10.05  239.0  126.0   

                                                                         \
                         Disposed with Benefit                            
                                         count percentage   mean median   
office                                                                    
Chief Secretary                         1127.0      55.00  306.0  222.0   
Collector                               4913.0      17.06  152.0  104.0   
DG & IG Police                           100.0       5.63   84.0   56.0   
Departments                             1742.0      12.33  182.0  149.0   
Governor                                   3.0       0.22  127.0  107.0   
NaN                                      739.0      24.22  137.0   87.0   
Office of Chief Minister                 758.0       3.84  396.0  392.0   
Superintendent of Police                 214.0       4.65  281.0   64.0   

                                                            \
                         Disposed                            
                            count percentage   mean median   
office                                                       
Chief Secretary             396.0      19.33  250.0  210.0   
Collector                 20143.0      69.97  162.0  107.0   
DG & IG Police             1600.0      90.09  156.0  106.0   
Departments                9702.0      68.68  249.0  194.0   
Governor                   1115.0      81.99  287.0  213.0   
NaN                        2164.0      70.93  187.0  107.0   
Office of Chief Minister  17007.0      86.22  288.0  237.0   
Superintendent of Police   3920.0      85.24  181.0  143.0   

                                                                           \
                         In Follow Up With Close                            
                                           count percentage   mean median   
office                                                                      
Chief Secretary                              0.0       0.00    0.0    0.0   
Collector                                   18.0       0.06   55.0   33.0   
DG & IG Police                               1.0       0.06  149.0  149.0   
Departments                                  9.0       0.06  113.0   64.0   
Governor                                     0.0       0.00    0.0    0.0   
NaN                                          0.0       0.00    0.0    0.0   
Office of Chief Minister                     0.0       0.00    0.0    0.0   
Superintendent of Police                     3.0       0.07   27.0   14.0   

                         2022-2023                           \
                           Discard                            
                             count percentage   mean median   
office                                                        
Chief Secretary             1118.0      30.08    6.0    1.0   
Collector                   8194.0      11.46   11.0    1.0   
DG & IG Police               173.0       6.03   10.0    2.0   
Departments                 4725.0      20.60   38.0    3.0   
Governor                     401.0      28.68   18.0    7.0   
NaN                          142.0       4.58  282.0  168.0   
Office of Chief Minister    5469.0      21.80    3.0    1.0   
Supe

In [21]:
pivot_pet_bucket_dept = petitioner_disposal_time_pivot(
    df_res, "dept")
pivot_pet_bucket_dept

2021-2022             \
                                                     Discard              
                                                       count percentage   
dept                                                                      
Agriculture & Farmers' Empowerment                      70.0       4.14   
Co-operation                                             7.0       2.82   
Commerce & Transport                                    12.0       3.57   
Electronics & Information Technology                     4.0       5.48   
Energy                                                  13.0       1.25   
Excise                                                   2.0       1.42   
Finance                                                 13.0       5.65   
Fisheries & Animal Resources Development                 1.0       0.53   
Food Supplies & Consumer Welfare                        42.0       1.83   
Forest, Environment and Climate Change Department        5.0       2.00   
General Administration & Public Grievance              156.0       8.64   
Handlooms ,Textiles & Handicrafts                        1.0       3.33   
Health & Family Welfare                                 78.0       2.58   
Higher Education                                        58.0      12.64   
Home department                                        172.0       2.38   
Housing & Urban Development                            123.0       2.20   
Industries                                               4.0       7.02   
Information & Public Relations                          10.0      17.24   
Labour & Employees' State Insurance                     12.0       3.05   
Law                                                      2.0       2.17   
Micro, Small & Medium Enterprise                         4.0      12.50   
Mission Shakti                                           0.0       0.00   
NaN                                                   7525.0      33.47   
Odia Language Literature & Culture                       2.0       1.54   
Panchayati Raj & Drinking Water                        250.0       1.58   
Parliamentary Affairs                                    0.0       0.00   
Planning & Convergence                                   1.0       2.50   
Public Enterprises                                       0.0       0.00   
Revenue & Disaster Management                          158.0       3.23   
Rural Development                                       42.0       8.84   
ST & SC Development, Minorities & Backward Clas...      49.0       9.63   
School & Mass Education                                 58.0       2.17   
Science & Technology                                     0.0       0.00   
Skill Development & Technical Education                  8.0       4.91   
Social Security & Empowerment of Persons with D...      69.0       4.61   
Sports & Youth Services                                  1.0       2.27   
Steel & Mines                                            6.0      10.53   
Tourism                                                  3.0       8.33   
Water Resources                                         14.0       2.53   
Women & Child Development                                6.0       1.16   
Works                                                   40.0      10.42   

                                                                  \
                                                                   
                                                     mean median   
dept                                                               
Agriculture & Farmers' Empowerment                   85.0   25.0   
Co-operation                                        251.0  243.0   
Commerce & Transport                                158.0  151.0   
Electronics & Information Technology                222.0  152.0   
Energy                                              188.0   69.0   
Excise                                               70.0   70.

In [22]:
pivot_pet_bucket_category = petitioner_disposal_time_pivot(
    df_res, "category")
pivot_pet_bucket_category


2021-2022                           \
                              Discard                            
                                count percentage   mean median   
category                                                         
Accident                          1.0       7.69  243.0  243.0   
Agriculture & Farming            24.0       2.86  199.0   62.0   
BSKY                              0.0       0.00    0.0    0.0   
CMRF                              3.0       0.58  411.0  460.0   
COVID-19                          6.0       4.14  235.0  147.0   
Culture                           0.0       0.00    0.0    0.0   
Disaster Management               0.0       0.00    0.0    0.0   
Education                        55.0       6.70  530.0  546.0   
Energy                           26.0       2.21  200.0  166.0   
Environment                       0.0       0.00    0.0    0.0   
Excise                            1.0       0.75  121.0  121.0   
Financial Assistance             17.0       2.46  214.0  187.0   
General                         247.0       2.51  222.0  166.0   
Health Care                      49.0       3.13  375.0  348.0   
Housing                         145.0       1.22  125.0   80.0   
ICDS                              2.0       1.14   50.0   50.0   
Infrastructure                   92.0       5.89  200.0  105.0   
Irrigation                        0.0       0.00    0.0    0.0   
Land Matters                     66.0       2.97  153.0  132.0   
Legal                             0.0       0.00    0.0    0.0   
Miscellaneous                   144.0       4.61  180.0  120.0   
NaN                            7699.0      31.77   49.0    2.0   
Pension/Retirement Benefits      30.0       2.61  243.0  125.0   
Police Case                      95.0       2.09  130.0  101.0   
Public Utility                   24.0       3.42  195.0  141.0   
Scheme &amp;amp; Benefits         0.0       0.00    0.0    0.0   
School & College                  7.0       2.77  559.0  781.0   
Service Matters                 111.0       3.98  241.0  153.0   
Social Welfare                  128.0       2.39  181.0   98.0   
Sports                            1.0       2.86  243.0  243.0   
Tourism                           0.0       0.00    0.0    0.0   
Traffic                           0.0       0.00    0.0    0.0   
Transport                         6.0       2.75  201.0  194.0   
Waste Management                  9.0       4.41  241.0  271.0   
Water Supply                     23.0       1.67  254.0  216.0   
Women Issues                      1.0       1.09  201.0  201.0   

                                                                            \
                            Disposed with Benefit                            
                                            count percentage   mean median   
category                                                                     
Accident                                      3.0      23.08  148.0  174.0   
Agriculture & Farming                        73.0       8.69  232.0  193.0   
BSKY                                          4.0       7.41  166.0   42.0   
CMRF                                        165.0      32.16  294.0  286.0   
COVID-19                                     28.0      19.31  227.0   99.0   
Culture                                       1.0       8.33   58.0   58.0   
Disaster Management                          24.0      27.91  106.0  100.0   
Education                                    83.0      10.11  294.0  110.0   
Energy                                      366.0      31.12  131.0   80.0   
Environment                                   6.0       5.45  201.0   46.0   
Excise                                        4.0       3.01  151.0  179.0   
Financial Assistance                         90.0      13.01  276.0  264.0   
General                                    2412.0      24.49  152.0   79.0   
Health Care                                 208.0      13.29 

In [23]:
pivot_pet_bucket_subcategory = petitioner_disposal_time_pivot(
    df_res, "subcategory")
pivot_pet_bucket_subcategory.head(10)

2021-2022                           \
                        Discard                            
                          count percentage   mean median   
subcategory                                                
AIDS                        0.0       0.00    0.0    0.0   
Aadhar Card Issues          0.0       0.00    0.0    0.0   
Absence of officials        0.0       0.00    0.0    0.0   
Absurd/Irrelevant           0.0       0.00    0.0    0.0   
Accidental Claims           0.0       0.00    0.0    0.0   
Admissions                  1.0       3.33  865.0  865.0   
Agriculture/Farming         3.0       2.03  153.0  160.0   
Ahar Yojana                 0.0       0.00    0.0    0.0   
Allegation/Corruption      19.0       3.85  123.0   38.0   
Anganwadi Centres           0.0       0.00    0.0    0.0   

                                                                               \
                      Disposed with Benefit                          Disposed   
                                      count percentage   mean median    count   
subcategory                                                                     
AIDS                                    0.0       0.00    0.0    0.0      0.0   
Aadhar Card Issues                      1.0       6.67  131.0  131.0     14.0   
Absence of officials                    0.0       0.00    0.0    0.0      0.0   
Absurd/Irrelevant                       1.0      20.00  152.0  152.0      4.0   
Accidental Claims                       0.0       0.00    0.0    0.0      5.0   
Admissions                              5.0      16.67  114.0  102.0     24.0   
Agriculture/Farming                    13.0       8.78  307.0  261.0    132.0   
Ahar Yojana                             1.0      20.00   17.0   17.0      4.0   
Allegation/Corruption                  32.0       6.49  195.0  164.0    442.0   
Anganwadi Centres                       0.0       0.00    0.0    0.0      0.0   

                                                                        \
                                               In Follow Up With Close   
                      percentage   mean median                   count   
subcategory                                                              
AIDS                        0.00    0.0    0.0                     0.0   
Aadhar Card Issues         93.33  198.0  161.0                     0.0   
Absence of officials        0.00    0.0    0.0                     0.0   
Absurd/Irrelevant          80.00  409.0  230.0                     0.0   
Accidental Claims         100.00  164.0  153.0                     0.0   
Admissions                 80.00  284.0  219.0                     0.0   
Agriculture/Farming        89.19  195.0  138.0                     0.0   
Ahar Yojana                80.00  163.0  202.0                     0.0   
Allegation/Corruption      89.66  282.0  233.0                     0.0   
Anganwadi Centres           0.00    0.0    0.0                     0.0   

                                             2022-2023                    \
                                               Discard                     
                      percentage mean median     count percentage   mean   
subcategory                                                                
AIDS                         0.0  0.0    0.0       0.0       0.00    0.0   
Aadhar Card Issues           0.0  0.0    0.0       0.0       0.00    0.0   
Absence of officials         0.0  0.0    0.0       0.0       0.00    0.0   
Absurd/Irrelevant            0.0  0.0    0.0       0.0       0.00    0.0   
Accidental Claims            0.0  0.0    0.0       0.0       0.00    0.0   
Admissions                   0.0  0.0    0.0       0.0       0.00    0.0   
Agriculture/Farming          0.0  0.0    0.0       4.0       0.96  181.0   
Ahar Yojana                  0.0  0.0    0.0       0.0       0.00    0.0   
Allegation/Corruption        0.0  0.0    0.0      28.0       1.92  158.0   
Anganwadi Centres    